In [ ]:
from agent_framework.ollama import OllamaChatClient

In [ ]:
import os
from pydantic import Field
from dotenv import load_dotenv
from typing import Annotated

import sys                                                                                                                              
sys.path.insert(0, "../")  # adjust depth based on notebook location to import utils       
from utils import stream_response

In [ ]:
load_dotenv()

In [ ]:
def web_search(
    query: Annotated[str, Field(description="Search query")],
) -> str:
    import requests
    
    api_key = os.getenv("OLLAMA_API_KEY")
    if not api_key:
        return "Error: OLLAMA_API_KEY environment variable not set"
    
    url = "https://ollama.com/api/web_search"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "query": query
    }
    
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        print(response.text)
        return response.text
    except requests.exceptions.RequestException as e:
        return f"Error fetching web content: {str(e)}"


In [ ]:
agent = OllamaChatClient(model=os.getenv("OLLAMA_CHAT_MODEL_ID")).as_agent(
        name="SearchAgent",
        instructions="You are my assistant. Answer the questions based on the search engine.",
        tools=[web_search],
)

In [ ]:
query = "What's ollama?"
response = await stream_response(agent.run(query, stream=True))